In [6]:
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoImageProcessor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
import os

def load_custom_dataset(root_dir):
    data = []
    for label in os.listdir(root_dir):
        label_dir = os.path.join(root_dir, label)
        if os.path.isdir(label_dir):
            for image_name in os.listdir(label_dir):
                image_path = os.path.join(label_dir, image_name)
                data.append({"image": image_path, "label": label})
    return data

root_dir = "D:\dataset_SIDDHI"  # Replace with your dataset path
data = load_custom_dataset(root_dir)
print(data[0])

# Split the data into train and test sets
train_data, test_data = train_test_split(data, test_size=0.2, stratify=[item['label'] for item in data])
val_data, test_data = train_test_split(test_data, test_size=0.5, stratify=[item['label'] for item in test_data])

# Convert to Hugging Face dataset
train_dataset = DatasetDict({"train": train_data})
test_dataset = DatasetDict({"test": test_data})
val_dataset = DatasetDict({"val": val_data})

data_dict_train = {
    'image': [item['image'] for item in train_dataset['train']],
    'label': [item['label'] for item in train_dataset['train']]
}

data_dict_test = {
    'image': [item['image'] for item in test_dataset['test']],
    'label': [item['label'] for item in test_dataset['test']]
}

data_dict_val = {
    'image': [item['image'] for item in val_dataset['val']],
    'label': [item['label'] for item in val_dataset['val']]
}

# Create the dataset
train_dataset = Dataset.from_dict(data_dict_train)
test_dataset = Dataset.from_dict(data_dict_test)
val_dataset = Dataset.from_dict(data_dict_val)

# Check the dataset
print(train_dataset)
print(test_dataset)
print(val_dataset)

# Preprocessing function
processor = AutoImageProcessor.from_pretrained("google/vit-hybrid-base-bit-384")

def preprocess(example_batch):
    images = [plt.imread(x) for x in example_batch['image']]
    images = [transforms.ToTensor()(img) for img in images]
    d = {}
    d["image"] = processor(images)['pixel_values']
    d["label"] = example_batch["label"]
    return d

# Apply the preprocessing function to the datasets
train_dataset = train_dataset.map(preprocess, batched=True, batch_size = 64)
test_dataset = test_dataset.map(preprocess, batched=True, batch_size = 64)
val_dataset = val_dataset.map(preprocess, batched=True, batch_size = 64)

#If you want to save
train_dataset.save_to_disk('./train_dataset')
val_dataset.save_to_disk('./val_dataset')
test_dataset.save_to_disk('./test_dataset')

{'image': 'D:\\dataset_SIDDHI\\CATARACT\\20240301100033_right_anterior9447.png', 'label': 'CATARACT'}
Dataset({
    features: ['image', 'label'],
    num_rows: 7203
})
Dataset({
    features: ['image', 'label'],
    num_rows: 901
})
Dataset({
    features: ['image', 'label'],
    num_rows: 900
})


Map:   0%|          | 0/7203 [00:00<?, ? examples/s]

Map:   0%|          | 0/901 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Saving the dataset (0/26 shards):   0%|          | 0/7203 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/900 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/901 [00:00<?, ? examples/s]